In [1020]:
import pytreenet as ptn
from copy import deepcopy
import numpy as np
import cmath
from scipy.linalg import expm
#np.random.seed(57643)

# Initialize vectorized rho


In [1021]:
def product_state(ttn, bond_dim=0 , physical_dim= 2):
    product_state = deepcopy(ttn)
    #A = np.array([1/np.sqrt(2),1/np.sqrt(2)])
    A = np.array([0,1])
    alpha = 1
    for node_id in product_state.nodes.keys():
        n = product_state.tensors[node_id].ndim - 1
        tensor = A.reshape((1,) * n + (physical_dim,))
        T = np.pad(tensor, n*((bond_dim, bond_dim),) + ((0, 0),))
        product_state.tensors[node_id] = T
        product_state.nodes[node_id].link_tensor(T)  
    return product_state



# local physical dimension
d = 2

shapes = {
    (0, 0): (3, 5, 6, d),
    (0, 1): (3, 2, d),
    (1, 0): (5, d),
    (1, 1): (2, d)
}


sites = {
    (i, j): ptn.random_tensor_node(shapes[(i, j)], identifier=f"Site({i},{j})") for i in range(2) for j in range(2)
}

vectorized_pho = ptn.TreeTensorNetworkState()

vectorized_pho.add_root(sites[(0, 0)][0], sites[(0, 0)][1])

connections = [
    ((0, 0), (0, 1), 0, 0),
    ((0, 0), (1, 0), 1, 0),
    ((0, 1), (1, 1), 1, 0)]


for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Site({parent[0]},{parent[1]})"
    child_id = f"Site({child[0]},{child[1]})"
    vectorized_pho.add_child_to_parent(sites[child][0], sites[child][1], child_leg, parent_id, parent_leg)

vectorized_pho = product_state(vectorized_pho , bond_dim=4, physical_dim = d)

nodes = {
    (i, j): (ptn.Node(tensor=vectorized_pho.tensors[f"Site({i},{j})"] , identifier=f"Node({i},{j})"), vectorized_pho.tensors[f"Site({i},{j})"]) for i in range(2) for j in range(2)
}

vectorized_pho.add_child_to_parent(nodes[(0,0)][0], nodes[(0,0)][1], 2, "Site(0,0)", 2)

connections = [
    ((0, 0), (0, 1), 1, 0),
    ((0, 0), (1, 0), 2, 0),
    ((0, 1), (1, 1), 1, 0)]

for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Node({parent[0]},{parent[1]})"
    vectorized_pho.add_child_to_parent(nodes[child][0], nodes[child][1], child_leg, parent_id, parent_leg)

# Effective Hamiltonian matrix (Directly from fromula)

In [1022]:
def liouville_matrix(t, U, gamma, m ,L, Lx, Ly, d):
    N = Lx * Ly  # Total number of sites
    dim = d**N  # Total Hilbert space dimension
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)

    # Initialize Hamiltonian
    H = np.zeros((dim, dim), dtype=complex)
    
    #################### Build Hamiltonian ###########################
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            neighbors = get_neighbors_Site(x, y, Lx, Ly)
            for neighbor in neighbors:
                term = np.eye(1)
                for x in range(Lx):
                    for y in range(Ly):
                        current_site_prime = f"Site({x},{y})"
                        if current_site_prime == current_site:
                            term = np.kron(term, creation_op)
                        elif current_site_prime == neighbor:
                            term = np.kron(term, annihilation_op)
                        else:
                            term = np.kron(term, np.eye(d))
                H += -t * (term + term.conj().T) 

    # On-site interaction 
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            term = np.eye(1)

            for x in range(Lx):
                for y in range(Ly):
                    current_site_prime = f"Site({x},{y})"
                    if current_site_prime == current_site:
                        interaction = 0.5 * U * number_op @ (number_op - np.eye(d)) 
                        chemical_potential = -m * number_op 
                        term = np.kron(term, interaction + chemical_potential)
                    else:
                        term = np.kron(term, np.eye(d))

            H += term
    ############################################################
    #################### Build liouville_matrix ################
            I = np.eye(dim)
            # Term 1: -i(H ⊗ I - I ⊗ H^T)
            unitary = -1j * (np.kron(H, I) - np.kron(I, H.T))
    
            # lowering operators on each site
            creation_op, annihilation_op, _ = ptn.bosonic_operators(d)
            lindblad_ops = []
            for i in range(Lx * Ly):
                op = np.eye(1)
                for j in range(Lx * Ly):
                    if j == i:
                        op = np.kron(op, annihilation_op)
                    else:
                        op = np.kron(op, np.eye(d))
                lindblad_ops.append(op)

            nonunitary = np.zeros((dim**2, dim**2), dtype=complex)
            for L in lindblad_ops:
                # L ⊗ L*
                LL = np.kron(L, L.conj())
                # (L^†L) ⊗ I
                LdL_I = np.kron(L.conj().T @ L, I)
                # I ⊗ (L^T L*)
                I_LtLs = np.kron(I, (L.conj().T @ L).T)
                
                nonunitary += gamma * (LL - 0.5 * LdL_I - 0.5 * I_LtLs)

    return unitary + nonunitary 


In [1023]:
t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op
gamma = 1

hamiltonian_matrix = liouville_matrix(t, U, gamma, m ,L, 2, 2, d)

# build operator N with Hamiltonian.to_tensor

In [1024]:
# build operator from O = Hamiltonian.to_tensor
# O.operator legs : (all in legs, all out legs) according to O.node_identifiers
# N : ( ket_in , bra_in , ket_out , bra_out )

def Number_op_total(Lx, Ly, dim=2):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(dim)
    conversion_dict = {"n": number_op , f"I{dim}": np.eye(dim)}
    terms = []
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "n"}))

    return ptn.Hamiltonian(terms, conversion_dict)
N = Number_op_total(2, 2, 2)
N = N.pad_with_identities(vectorized_pho, symbolic= True)
N_tn = N.to_tensor(vectorized_pho)
N = N_tn.operator
N_order = N_tn.node_identifiers
print( "N.shape" , N.shape) 
print("N_order", N_order)
N = N.reshape((256,256))


N.shape (2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2)
N_order ['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)']


# Build initial rho vector

In [1025]:
def get_permutation_tuple(list1, list2):
    index_map = {element: index for index, element in enumerate(list1)}
    permutation = tuple(index_map[element] for element in list2)
    return permutation

def get_initial_rho(vectorized_pho , initial_order):
    ttn_cont , ttn_order = vectorized_pho.completely_contract_tree(to_copy = True)
    print("ttn_order",ttn_order)
    perm  = get_permutation_tuple(ttn_order, initial_order)
    print("perm",perm)
    rho = np.transpose(ttn_cont, perm)
    rho = np.reshape(rho, (256))
    return rho
    
print("initial_order", initial_order)   
initial_rho = get_initial_rho(vectorized_pho , initial_order)
# rho : ( ket_in x bra_in )

initial_order ['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)']
ttn_order ['Site(0,0)', 'Site(0,1)', 'Site(1,1)', 'Site(1,0)', 'Node(0,0)', 'Node(0,1)', 'Node(1,1)', 'Node(1,0)']
perm (0, 1, 3, 2, 4, 5, 7, 6)


# Evaluate exact expectation value

In [1026]:
def exp_val(rho,op):
    op_rho = np.tensordot(op, rho, axes = ((1),(0,)))
    tr_op_rho = np.trace(op_rho.reshape((16,16)))
    tr_tho = np.trace(rho.reshape((16,16))) 
    return tr_op_rho / tr_tho
exp_val(initial_rho, N)

4.0

# Run Exact solution

In [1027]:
time_evolution_operator = expm((0.01) * hamiltonian_matrix)
rho = initial_rho
for _ in range(100):
    rho = np.tensordot(time_evolution_operator, rho, axes = ((1,),(0)))
    print(exp_val(rho,N))

(3.9601993349966724+7.426025294966965e-27j)
(3.9207946932270215+1.0213787316084585e-25j)
(3.881782134194032+2.6812539634908347e-25j)
(3.843157756609293+4.909583698866299e-25j)
(3.8049176980028565+7.576668039033238e-25j)
(3.7670581343369953+1.056629116703022e-24j)
(3.729575279623793+1.3774676947933981e-24j)
(3.692465385546543+1.710951393514922e-24j)
(3.6557247410849123+2.0489045573617327e-24j)
(3.6193496721438385+2.3841221631446017e-24j)
(3.5833365411861133+2.7102907118829614e-24j)
(3.54768174686863+3.0219145167998948e-24j)
(3.512381723682245+3.3142470550970445e-24j)
(3.477432941595223+3.583227070370314e-24j)
(3.4428319057002312+3.825419130649864e-24j)
(3.4085751558648454+4.03795836416856e-24j)
(3.3746592663855344+4.218499111131908e-24j)
(3.3410808456450884+4.365167245037783e-24j)
(3.3078365357734487+4.476515931511629e-24j)
(3.2749230123119277+4.5514846062463306e-24j)
(3.2423369838807483+4.589360966482483e-24j)
(3.210075191849914+4.589745782618696e-24j)
(3.1781344100133357+4.55252034797

# Compare with solution from Qutip Library (MATCHED)

In [1028]:
from qutip import *
import numpy as np

# Define parameters
t = 0.4  # Hopping strength
U = 0.8  # On-site interaction strength
m = 0.4  # Chemical potential
gamma_relax = 1  # Relaxation rate

# Reduced lattice dimensions
Nx = 2  # Number of sites along x-direction
Ny = 2  # Number of sites along y-direction
T = Nx * Ny  # Total number of sites

# Reduced maximum number of bosons per site
nmax = 1

# Precompute the operators for each site
a_list = []
adag_list = []
n_list = []
si = qeye(nmax + 1)  # Identity operator for a single site
for n in range(T):
    op_list = [si] * T
    op_list[n] = destroy(nmax + 1)
    a_op = tensor(op_list)
    a_list.append(a_op)
    adag_list.append(a_op.dag())
    n_list.append(a_op.dag() * a_op)


# Function to map 2D lattice coordinates (i, j) to a site index
def site(i, j):
    return i + j * Nx

# Initialize the Hamiltonian
H = 0

# Build the Hamiltonian by summing over sites
for i in range(Nx):
    for j in range(Ny):
        n = site(i, j)
        H += 0.5 * U * n_list[n] * (n_list[n] - 1) - m * n_list[n]
        if i < Nx - 1:
            n_right = site(i + 1, j)
            H += -t * (adag_list[n] * a_list[n_right] + adag_list[n_right] * a_list[n])
        if j < Ny - 1:
            n_up = site(i, j + 1)
            H += -t * (adag_list[n] * a_list[n_up] + adag_list[n_up] * a_list[n])

# Initial state: product state of maximum occupation

psi0 = tensor([basis(nmax + 1, 1) for _ in range(T)]).unit()
#psi0 = tensor([(basis(nmax + 1, 0) + basis(nmax + 1, 1)).unit() for _ in range(T)])
alpha = 1
#psi0 = 1j *tensor([coherent(nmax + 1, alpha) for _ in range(T)])
# Reduced simulation time and increased time step
total_time = 1  # Total time in seconds
time_step = 0.01  # Time step in seconds
tlist = np.arange(0, total_time + time_step, time_step)

# Define collapse operators (for the Lindblad equation)
custom_matrix = Qobj([[0, 1], [0, 0]])
jump_operator = []
si = qeye(nmax + 1) 
for n in range(T):
  op_list = [si] * T  # Create a list of identity operators
  op_list[n] = custom_matrix  # Replace the n-th site with the custom matrix
  custom_op = tensor(op_list)  # Create the tensor product
  jump_operator.append(custom_op)

c_ops = [np.sqrt(gamma_relax) * a for a in a_list]


# Observables to calculate - total particle number
N_total = sum(n_list)

# Solve the Schrödinger equation (more efficient for this case)
result = mesolve(H, psi0, tlist, c_ops, [N_total])

# Extract expectation values
total_number = result.expect[0]

# Print results
print("Time evolution of total particle number:")
for t, n in zip(tlist, total_number):
    print(f"Time: {t:.2f}, Total number: {n:.4f}")

Time evolution of total particle number:
Time: 0.00, Total number: 4.0000
Time: 0.01, Total number: 3.9602
Time: 0.02, Total number: 3.9208
Time: 0.03, Total number: 3.8818
Time: 0.04, Total number: 3.8432
Time: 0.05, Total number: 3.8049
Time: 0.06, Total number: 3.7671
Time: 0.07, Total number: 3.7296
Time: 0.08, Total number: 3.6925
Time: 0.09, Total number: 3.6557
Time: 0.10, Total number: 3.6193
Time: 0.11, Total number: 3.5833
Time: 0.12, Total number: 3.5477
Time: 0.13, Total number: 3.5124
Time: 0.14, Total number: 3.4774
Time: 0.15, Total number: 3.4428
Time: 0.16, Total number: 3.4086
Time: 0.17, Total number: 3.3747
Time: 0.18, Total number: 3.3411
Time: 0.19, Total number: 3.3078
Time: 0.20, Total number: 3.2749
Time: 0.21, Total number: 3.2423
Time: 0.22, Total number: 3.2101
Time: 0.23, Total number: 3.1781
Time: 0.24, Total number: 3.1465
Time: 0.25, Total number: 3.1152
Time: 0.26, Total number: 3.0842
Time: 0.27, Total number: 3.0535
Time: 0.28, Total number: 3.0231
Ti

# Build Effective Hamiltonian with State Diagram

In [1029]:
def get_neighbors_Site(x, y, Lx, Ly):
  neighbors = []
  
  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Site({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Site({x},{y+1})")
  
  return neighbors

def get_neighbors_Node(x, y, Lx, Ly):
  neighbors = []

  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Node({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Node({x},{y+1})")
  
  return neighbors

def Liouville(t, U, gamma, m, L, Lx, Ly, d):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
    
    conversion_dict = {
        "b^dagger": creation_op,
        "b": annihilation_op,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "it * b^dagger": t*1j * creation_op,
        "it * b": t*1j * annihilation_op,
        "-iU * n * (n - 1)": -U*1j * number_op @ (number_op - np.eye(d)),
        "im*n": m*1j*number_op
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            neighbors = get_neighbors_Site(x, y, Lx, Ly)            
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "it * b^dagger", neighbor: "b"}))
                terms.append(ptn.TensorProduct({current_site: "it * b", neighbor: "b^dagger"}))
                

    
    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-iU * n * (n - 1)"}))

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "im*n"}))        
    
    H1 = ptn.Hamiltonian(terms, conversion_dict)
    
    conversion_dict = {
        "b^dagger.T": creation_op.T,
        "b.T": annihilation_op.T,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "-it * b^dagger.T": -t*1j * creation_op.T,
        "-it * b.T": -t*1j * annihilation_op.T,
        "iU * n * (n - 1).T": (U*1j * number_op @ (number_op - np.eye(d))).T,
        "-im*n.T": -m*1j* number_op.T
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            neighbors = get_neighbors_Node(x, y, Lx, Ly)
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "-it * b^dagger.T", neighbor: "b.T"}))
                terms.append(ptn.TensorProduct({current_site: "-it * b.T", neighbor: "b^dagger.T"}))

    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "iU * n * (n - 1).T"}))    

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-im*n.T"}))
            
    H2 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H2)

        
    conversion_dict = {    
    "L": np.sqrt(gamma) * L,
    "L^dagger.T": np.sqrt(gamma) * L.conj(),
    "-1/2 (L^dagger @ L) " : -1/2 * gamma * L.conj().T @ L,
    "-1/2 (L^dagger @ L).T": -1/2 * gamma * (L.conj().T @ L).T}
    terms = []
    for x in range(Lx):
        for y in range(Ly):
            out_site = f"Node({x},{y})"
            in_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({in_site: "L" , out_site: "L^dagger.T"}))
            terms.append(ptn.TensorProduct({in_site: "-1/2 (L^dagger @ L) "}))
            terms.append(ptn.TensorProduct({out_site: "-1/2 (L^dagger @ L).T"}))

    H3 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H3)
    return H1

vectorized_pho = ptn.adjust_bra_to_ket(vectorized_pho)

# METHOD 1 
# Effective Hamiltonian with Hamiltonian.to_matrix

In [1030]:
t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op
gamma = 1

L = Liouville(t, U, gamma, m ,L, 2, 2, d)
L = L.pad_with_identities(vectorized_pho, symbolic= True)
L = L.to_matrix(vectorized_pho)
hamiltonian_matrix_1 = L.operator
ham2_order = L.node_identifiers
print( "hamiltonian_matrix_2.shape" , hamiltonian_matrix_1.shape) 
print("ham2_order", ham2_order)

# Hamiltonian.to_matix legs : (in_legs, out_legs)
hamiltonian_matrix_1 = hamiltonian_matrix_1.T

hamiltonian_matrix_2.shape (256, 256)
ham2_order ['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)']


# Run Exact solution 
# (NOT MATCHED)

In [1031]:
time_evolution_operator = expm((0.01) * hamiltonian_matrix_1)
rho = initial_rho
for _ in range(100):
    rho = np.tensordot(time_evolution_operator, rho, axes = ((1,),(0)))
    print(exp_val(rho,N))

(3.9601993349966724+4.298977354807626e-22j)
(3.9207946932270215+3.3294241220187438e-21j)
(3.881782134194032+1.0878673875700384e-20j)
(3.843157756609293+2.4965734493426313e-20j)
(3.8049176980028565+4.721127489031063e-20j)
(3.7670581343369953+7.899143604664394e-20j)
(3.729575279623793+1.2145912880890995e-19j)
(3.692465385546543+1.755638379036039e-19j)
(3.6557247410849123+2.420700254732451e-19j)
(3.6193496721438385+3.215742220137934e-19j)
(3.5833365411861133+4.1452088746418893e-19j)
(3.54768174686863+5.212171203548931e-19j)
(3.5123817236822457+6.418462883494718e-19j)
(3.477432941595223+7.76480649195624e-19j)
(3.4428319057002312+9.25093027024961e-19j)
(3.4085751558648445+1.0875676050947651e-18j)
(3.374659266385535+1.2637098924323007e-18j)
(3.3410808456450876+1.4532559184166017e-18j)
(3.307836535773449+1.655880706098453e-18j)
(3.2749230123119273+1.871206072009289e-18j)
(3.2423369838807483+2.0988077973322325e-18j)
(3.2100751918499135+2.3382222125939915e-18j)
(3.178134410013336+2.588952235477

# METHOD 2 
# Effective Hamiltonian with Hamiltonian.to_tensor

In [1032]:
t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op
gamma = 1

L = Liouville(t, U, gamma, m ,L, 2, 2, d)
L = L.pad_with_identities(vectorized_pho, symbolic= True)
L = L.to_tensor(vectorized_pho)
L_tn = L.operator
ham3_order = L.node_identifiers
print( "L_tn.shape" , L_tn.shape) 
print("ham3_order", ham3_order)

hamiltonian_matrix_2 = L_tn.reshape((256,256))
hamiltonian_matrix_2 = hamiltonian_matrix_2.T

L_tn.shape (2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2)
ham3_order ['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)']


# Run Exact solution 
# (NOT MATCHED)

In [1033]:
time_evolution_operator = expm((0.01) * hamiltonian_matrix_2)
rho = initial_rho
for _ in range(100):
    rho = np.tensordot(time_evolution_operator, rho, axes = ((1,),(0)))
    print(exp_val(rho,N))

(3.9601993349966724+4.298977354807626e-22j)
(3.9207946932270215+3.3294241220187438e-21j)
(3.881782134194032+1.0878673875700384e-20j)
(3.843157756609293+2.4965734493426313e-20j)
(3.8049176980028565+4.721127489031063e-20j)
(3.7670581343369953+7.899143604664394e-20j)
(3.729575279623793+1.2145912880890995e-19j)
(3.692465385546543+1.755638379036039e-19j)
(3.6557247410849123+2.420700254732451e-19j)
(3.6193496721438385+3.215742220137934e-19j)
(3.5833365411861133+4.1452088746418893e-19j)
(3.54768174686863+5.212171203548931e-19j)
(3.5123817236822457+6.418462883494718e-19j)
(3.477432941595223+7.76480649195624e-19j)
(3.4428319057002312+9.25093027024961e-19j)
(3.4085751558648445+1.0875676050947651e-18j)
(3.374659266385535+1.2637098924323007e-18j)
(3.3410808456450876+1.4532559184166017e-18j)
(3.307836535773449+1.655880706098453e-18j)
(3.2749230123119273+1.871206072009289e-18j)
(3.2423369838807483+2.0988077973322325e-18j)
(3.2100751918499135+2.3382222125939915e-18j)
(3.178134410013336+2.588952235477

# METHOD 3
# Effective hamiltonian with completely contracting the N TTNO and the Liouville TTNO

In [1034]:
# ALL TENSORS ARE PERMUTED TO THIS ORDER
initial_order = ['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)']

In [1035]:
def get_op_matrix(op_tensor , op_tensor_order , initial_order):
    perm  = get_permutation_tuple(op_tensor_order,initial_order)
    perm_prime = tuple(j for i in perm for j in (2*i, 2*i+1))
    print("perm_prime",perm_prime)
    op = np.transpose(op_tensor, perm_prime)
    perm_double_prime = tuple (range(0,16,2)) + tuple(range(1,17,2))
    print("perm_double_prime",perm_double_prime)
    op = np.transpose(op, perm_double_prime)
    op_matrix = np.reshape(op, (256,256))
    return op_matrix

N = Number_op_total(2, 2, 2)
N = N.pad_with_identities(vectorized_pho, symbolic= True)
N_ttno = ptn.TTNO.from_hamiltonian(N , vectorized_pho)
N_tn , N_order = N_ttno.completely_contract_tree(to_copy = True)
print("initial_order", initial_order)
print("N_order" , N_order)

N = get_op_matrix(N_tn , N_order , initial_order)


initial_order ['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)']
N_order ['Site(0,0)', 'Node(0,0)', 'Node(0,1)', 'Node(1,1)', 'Node(1,0)', 'Site(0,1)', 'Site(1,1)', 'Site(1,0)']
perm_prime (0, 1, 10, 11, 14, 15, 12, 13, 2, 3, 4, 5, 8, 9, 6, 7)
perm_double_prime (0, 2, 4, 6, 8, 10, 12, 14, 1, 3, 5, 7, 9, 11, 13, 15)


In [1036]:
t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op
gamma = 1

H1 = Liouville(t, U, gamma, m ,L, 2, 2, d)
L_fancy = ptn.TTNO.from_hamiltonian(H1, vectorized_pho)
ham_tensor , ham_tensor_order = L_fancy.completely_contract_tree()
print("initial order", initial_order)
print("ham_tensor_order" , ham_tensor_order)

def get_hamiltonian_matrix(ham_tensor , ham_tensor_order , initial_order):
    perm  = get_permutation_tuple(ham_tensor_order,initial_order)
    perm_prime = tuple(j for i in perm for j in (2*i, 2*i+1))
    print("perm_prime",perm_prime)
    ham = np.transpose(ham_tensor, perm_prime)
    perm_double_prime = tuple (range(0,16,2)) + tuple(range(1,17,2))
    print("perm_double_prime",perm_double_prime)
    ham = np.transpose(ham, perm_double_prime)
    ham_matrix = np.reshape(ham, (256,256))
    return ham_matrix

hamiltonian_matrix_3 = get_hamiltonian_matrix(ham_tensor , ham_tensor_order, initial_order)
# (out : act on <ket(sites)| x <bra(Nodes)| , in : act on |ket(sites)> x |bra(Nodes)>)

initial order ['Site(0,0)', 'Site(0,1)', 'Site(1,0)', 'Site(1,1)', 'Node(0,0)', 'Node(0,1)', 'Node(1,0)', 'Node(1,1)']
ham_tensor_order ['Site(0,0)', 'Node(0,0)', 'Node(0,1)', 'Node(1,1)', 'Node(1,0)', 'Site(0,1)', 'Site(1,1)', 'Site(1,0)']
perm_prime (0, 1, 10, 11, 14, 15, 12, 13, 2, 3, 4, 5, 8, 9, 6, 7)
perm_double_prime (0, 2, 4, 6, 8, 10, 12, 14, 1, 3, 5, 7, 9, 11, 13, 15)


# Run Exact solution 
# (NOT MATCHED)

In [1037]:
time_evolution_operator = expm((0.01) * hamiltonian_matrix_3)
rho = initial_rho
for _ in range(100):
    rho = np.tensordot(time_evolution_operator, rho, axes = ((1,),(0)))
    print(exp_val(rho,N))

(3.9601993349966724-4.298977354807626e-22j)
(3.9207946932270215-3.3294241220187438e-21j)
(3.881782134194032-1.0878673875700384e-20j)
(3.843157756609293-2.4965734493426313e-20j)
(3.8049176980028565-4.721127489031063e-20j)
(3.7670581343369953-7.899143604664394e-20j)
(3.729575279623793-1.2145912880890995e-19j)
(3.692465385546543-1.755638379036039e-19j)
(3.6557247410849123-2.420700254732451e-19j)
(3.6193496721438385-3.215742220137934e-19j)
(3.5833365411861133-4.1452088746418893e-19j)
(3.54768174686863-5.212171203548931e-19j)
(3.5123817236822457-6.418462883494718e-19j)
(3.477432941595223-7.76480649195624e-19j)
(3.4428319057002312-9.25093027024961e-19j)
(3.4085751558648445-1.0875676050947651e-18j)
(3.374659266385535-1.2637098924323007e-18j)
(3.3410808456450876-1.4532559184166017e-18j)
(3.307836535773449-1.655880706098453e-18j)
(3.2749230123119273-1.871206072009289e-18j)
(3.2423369838807483-2.0988077973322325e-18j)
(3.2100751918499135-2.3382222125939915e-18j)
(3.178134410013336-2.588952235477